In [26]:
# A6.0 - Load saved modeling dataset

import pandas as pd
from pathlib import Path

# Automatically locate the project root
ROOT = Path.cwd().resolve()

while not (ROOT / "data").is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Project root not found")
    ROOT = ROOT.parent

# Load modeling dataset
modeling_data = pd.read_csv(
    ROOT / "data/processed/modeling_dataset.csv",
    parse_dates=["snapshot_date"]
)

print("Project root:", ROOT)
print("Dataset shape:", modeling_data.shape)

print("\nTarget distribution:")
print(modeling_data["future_investigation"].value_counts())

display(modeling_data.head())


Project root: /Users/xinyisong/Desktop/DATA1030-Midterm-Xinfinity
Dataset shape: (2480, 24)

Target distribution:
future_investigation
0    2123
1     357
Name: count, dtype: int64


,make,model_family,model_year,snapshot_date,future_investigation,complaints_last_12m,crash_count_12m,fire_count_12m,injuries_12m,deaths_12m,...,complaint_trend_3m,engine_complaints_12m,brake_complaints_12m,electrical_complaints_12m,powertrain_complaints_12m,airbag_complaints_12m,component_diversity_12m,avg_reporting_delay_12m,median_reporting_delay_12m,valid_delay_count_12m
0,AUDI,Q5,2015,2016-01-01,0,2,1.0,0.0,1.0,0.0,...,0.000000,0,0,0,0,0,2,6.000000,6.0,2
1,AUDI,Q5,2016,2016-01-01,0,1,0.0,0.0,0.0,0.0,...,NaN,0,0,1,0,0,1,30.000000,30.0,1
2,BMW,3 SERIES,2015,2016-01-01,0,9,0.0,0.0,0.0,0.0,...,2.400000,2,2,2,2,0,9,19.000000,2.0,9
3,BMW,3 SERIES,2016,2016-01-01,0,0,0.0,0.0,0.0,0.0,...,NaN,0,0,0,0,0,0,NaN,NaN,0
4,FORD,ESCAPE,2015,2016-01-01,1,28,3.0,0.0,2.0,0.0,...,1.666667,7,4,7,1,2,14,39.571429,17.5,28


In [2]:

# A6.1 - Final Time-Based Split
# Train / Vali / Test with temporal purging

import pandas as pd

split_data = modeling_data.copy()

split_data["label_end"] = (
    split_data["snapshot_date"] +
    pd.DateOffset(months=12)
)

test_start = pd.Timestamp("2024-01-01")

# Train pool, temporal gap, and final test
split_data["split"] = "gap"

split_data.loc[
    split_data["label_end"] <= test_start,
    "split"
] = "train"

split_data.loc[
    split_data["snapshot_date"] >= test_start,
    "split"
] = "test"

# Create datasets
train_data = split_data[
    split_data["split"] == "train"
].copy()

test_data = split_data[
    split_data["split"] == "test"
].copy()

gap_data = split_data[
    split_data["split"] == "gap"
].copy()

# Summarize
split_summary = (
    split_data.groupby("split")
    .agg(
        samples=("future_investigation", "size"),
        positives=("future_investigation", "sum")
    )
)

split_summary["negatives"] = (
    split_summary["samples"] -
    split_summary["positives"]
)

split_summary["positive_rate"] = (
    split_summary["positives"] /
    split_summary["samples"]
).round(4)

print("===== SPLIT SUMMARY =====")
display(split_summary)

# Check temporal leakage
assert train_data["label_end"].max() <= \
       test_data["snapshot_date"].min()

assert len(split_data) == 2480

print("Time-based split validation passed!")


===== SPLIT SUMMARY =====


,samples,positives,negatives,positive_rate
split,,,,
gap,270,31,239,0.1148
test,720,29,691,0.0403
train,1490,297,1193,0.1993


Time-based split validation passed!


In [3]:

# A6.2 - Create purged expanding-window CV folds

import pandas as pd

# Validation start dates
validation_starts = [
    pd.Timestamp("2020-01-01"),
    pd.Timestamp("2021-01-01"),
    pd.Timestamp("2022-01-01")
]

cv_folds = []

for fold_id, vali_start in enumerate(validation_starts, start=1):

    vali_end = vali_start + pd.DateOffset(years=1)

    # Training labels must finish before validation begins
    train_mask = (
        train_data["label_end"] <= vali_start
    )

    # Validation snapshots within the selected year
    vali_mask = (
        (train_data["snapshot_date"] >= vali_start) &
        (train_data["snapshot_date"] < vali_end)
    )

    train_fold = train_data.loc[train_mask].copy()
    vali_fold = train_data.loc[vali_mask].copy()

    # Validate no overlap in prediction windows
    assert (
        train_fold["label_end"].max()
        <= vali_fold["snapshot_date"].min()
    )

    cv_folds.append((train_fold, vali_fold))

    print(f"\n===== FOLD {fold_id} =====")

    print(
        "Train:",
        len(train_fold),
        "samples,",
        int(train_fold["future_investigation"].sum()),
        "positives"
    )

    print(
        "Validation:",
        len(vali_fold),
        "samples,",
        int(vali_fold["future_investigation"].sum()),
        "positives"
    )

    print(
        "Last train label end:",
        train_fold["label_end"].max()
    )

    print(
        "Validation starts:",
        vali_fold["snapshot_date"].min()
    )

print("\nAll CV folds created and validated!")



===== FOLD 1 =====
Train: 410 samples, 51 positives
Validation: 240 samples, 29 positives
Last train label end: 2020-01-01 00:00:00
Validation starts: 2020-01-01 00:00:00

===== FOLD 2 =====
Train: 620 samples, 63 positives
Validation: 280 samples, 60 positives
Last train label end: 2021-01-01 00:00:00
Validation starts: 2021-01-01 00:00:00

===== FOLD 3 =====
Train: 870 samples, 111 positives
Validation: 320 samples, 111 positives
Last train label end: 2022-01-01 00:00:00
Validation starts: 2022-01-01 00:00:00

All CV folds created and validated!


In [4]:

# A6.3 - Save reproducible CV fold assignments

import pandas as pd
from pathlib import Path

project_dir = Path(
    ROOT
)

output_dir = project_dir / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

fold_records = []

for fold_id, (train_fold, vali_fold) in enumerate(
    cv_folds, start=1
):
    for role, subset in [
        ("train", train_fold),
        ("vali", vali_fold)
    ]:
        temp = subset[
            ["make", "model_family",
             "model_year", "snapshot_date"]
        ].copy()

        temp["fold"] = fold_id
        temp["role"] = role

        fold_records.append(temp)

cv_assignments = pd.concat(
    fold_records, ignore_index=True
)

# Save the full CV assignment table
cv_path = output_dir / "cv_fold_assignments.csv"
cv_assignments.to_csv(cv_path, index=False)

print("Saved:", cv_path)

print("\n===== CV ASSIGNMENT COUNTS =====")
display(
    cv_assignments.groupby(
        ["fold", "role"]
    ).size().unstack(fill_value=0)
)

# Verify assignments are unique within each fold
assert not cv_assignments.duplicated(
    ["fold", "make", "model_family",
     "model_year", "snapshot_date"]
).any()

print("\nCV assignments saved and validated!")


Saved: /Users/xinyisong/Desktop/DATA1030-Midterm-Xinfinity/data/processed/cv_fold_assignments.csv

===== CV ASSIGNMENT COUNTS =====


role,train,vali
fold,,
1,410,240
2,620,280
3,870,320



CV assignments saved and validated!


In [5]:

# A6.4.1 - Locate saved investigation data files

from pathlib import Path

project_dir = Path(
    ROOT
)

data_dir = project_dir / "data"

print("===== INVESTIGATION-RELATED FILES =====")

matches = [
    p for p in data_dir.rglob("*")
    if p.is_file()
    and any(
        keyword in p.name.lower()
        for keyword in ["investigation", "matched", "event", "inv"]
    )
]

for path in sorted(matches):
    print(path.relative_to(project_dir))

print("\nTotal matching files:", len(matches))


===== INVESTIGATION-RELATED FILES =====
data/processed/broad_events.csv
data/raw/Investigation/FLAT_INV.zip
data/raw/Investigation/INV.txt

Total matching files: 3


In [6]:

# A6.4.2 - Locate previous data preparation notebooks

from pathlib import Path

project_dir = Path(
    ROOT
)

notebooks = sorted(project_dir.rglob("*.ipynb"))

print("===== NOTEBOOK FILES =====")

for notebook in notebooks:
    print(notebook.relative_to(project_dir))

print("\nTotal notebooks:", len(notebooks))


===== NOTEBOOK FILES =====
notebooks/.ipynb_checkpoints/00_shared_preprocessing-checkpoint.ipynb
src/analysis/.ipynb_checkpoints/01_EDA-checkpoint.ipynb
src/analysis/.ipynb_checkpoints/02_EDA-checkpoint.ipynb
src/analysis/.ipynb_checkpoints/03_EDA-checkpoint.ipynb
src/analysis/01_EDA.ipynb
src/analysis/02_EDA.ipynb
src/analysis/03_EDA.ipynb
src/preprocessing/.ipynb_checkpoints/00_shared_preprocessing-checkpoint.ipynb
src/preprocessing/.ipynb_checkpoints/01_final_data_preparation-checkpoint.ipynb
src/preprocessing/.ipynb_checkpoints/02_modeling_and_splitting-checkpoint.ipynb
src/preprocessing/00_shared_preprocessing.ipynb
src/preprocessing/01_final_data_preparation.ipynb
src/preprocessing/02_modeling_and_splitting.ipynb

Total notebooks: 13


In [7]:

# A6.4.3 - Find investigation event code in old notebook

import json
from pathlib import Path

project_dir = Path(
    ROOT
)

notebook_path = (
    project_dir
    / "src/preprocessing/01_final_data_preparation.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    old_notebook = json.load(f)

keywords = [
    "inv_matched_final",
    "broad_events",
    "event_keys",
    "action_type"
]

for i, cell in enumerate(old_notebook["cells"]):

    source = "".join(cell.get("source", []))

    if cell.get("cell_type") != "code":
        continue

    matched = [
        word for word in keywords
        if word in source
    ]

    if matched:
        print(f"\n===== CELL {i} =====")
        print("Keywords:", matched)
        print(source[:1500])



===== CELL 13 =====
Keywords: ['action_type']

# A2.12 - Inspect action number prefixes

investigation_summary["action_type"] = (
    investigation_summary["action_number"].str[:2]
)

type_summary = (
    investigation_summary
    .groupby("action_type")
    .agg(
        unique_actions=("action_number", "nunique")
    )
    .reset_index()
    .sort_values("unique_actions", ascending=False)
)

display(type_summary)


===== CELL 17 =====
Keywords: ['inv_matched_final']

# A2.18 - Re-match investigations with expanded vehicle groups

expanded_match = expanded_groups.rename(
    columns={"model": "model_family"}
)

matching_keys = ["make", "model_family", "model_year"]

inv_matched_final = inv_clean.merge(
    expanded_match,
    on=matching_keys,
    how="inner",
    validate="many_to_one"
)

print("Matched investigation rows:", len(inv_matched_final))
print("Unique action numbers:",
      inv_matched_final["action_number"].nunique())
print("Matched vehicle groups:",
      inv_matched_f

In [8]:

# A6.4.4 - Locate investigation matching dependencies

keywords = [
    "inv_clean =",
    "expanded_groups =",
    "inv_clean[",
    "expanded_groups[",
    "read_csv(",
    "read_table("
]

for i, cell in enumerate(old_notebook["cells"]):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if any(keyword in source for keyword in keywords):
        # Only inspect cells before matching is completed
        if i <= 17:
            print(f"\n===== CELL {i} =====")
            print(source[:2500])



===== CELL 1 =====

import pandas as pd
from pathlib import Path

# Find project root
ROOT = Path.cwd().resolve()

while not (ROOT / "data").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Project data folder not found")
    ROOT = ROOT.parent

# Load cleaned data
data_path = ROOT / "data" / "processed" / "complaints_clean.csv"

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())


===== CELL 4 =====

import pandas as pd
import zipfile

# Column names from the official NHTSA data dictionary
inv_columns = [
    "action_number",
    "make",
    "model",
    "model_year",
    "component",
    "manufacturer",
    "open_date",
    "close_date",
    "campaign_number",
    "subject",
    "summary"
]

# Inspect ZIP contents
with zipfile.ZipFile(zip_path, "r") as z:
    files = [
        name for name in z.namelist()
        if name.lower().endswith(".txt")
    ]

    print("Text files inside Z

In [9]:

# A6.4.5 - Rebuild and save broad investigation events

import pandas as pd
import zipfile
from pathlib import Path

ROOT = Path(
    ROOT
)

zip_path = ROOT / "data/raw/Investigation/FLAT_INV.zip"

inv_columns = [
    "action_number", "make", "model", "model_year",
    "component", "manufacturer", "open_date",
    "close_date", "campaign_number", "subject", "summary"
]

# 1. Load original investigation data
with zipfile.ZipFile(zip_path) as z:
    txt_files = [
        name for name in z.namelist()
        if name.lower().endswith(".txt")
    ]
    assert len(txt_files) == 1

    with z.open(txt_files[0]) as f:
        inv = pd.read_csv(
            f,
            sep="\t",
            header=None,
            names=inv_columns,
            dtype=str,
            encoding="latin-1"
        )

# 2. Standardize matching keys and dates
for col in ["make", "model"]:
    inv[col] = inv[col].astype("string").str.strip().str.upper()

inv["model_year"] = pd.to_numeric(
    inv["model_year"], errors="coerce"
)

inv["open_date"] = pd.to_datetime(
    inv["open_date"],
    format="%Y%m%d",
    errors="coerce"
)

inv["model_family"] = inv["model"].copy()

# 3. Reuse the original alias definitions
model_aliases = {
    "AUDI": {
        "Q5 HYBRID": "Q5",
        "Q5 PHEV": "Q5"
    },
    "BMW": {
        name: "3 SERIES"
        for name in [
            "320I", "320XI", "328I", "328XI",
            "328D", "328XD", "330I", "330E",
            "335I", "335XI", "340I",
            "ACTIVEHYBRID 3", "M340I"
        ]
    },
    "MERCEDES-BENZ": {
        name: "C-CLASS"
        for name in [
            "C250", "C300", "C 300", "C350",
            "C350E", "C400", "C450",
            "AMG C43", "AMG C63", "AMG C63S"
        ]
    },
    "TOYOTA": {
        "CAMRY HYBRID": "CAMRY"
    }
}

for make, mapping in model_aliases.items():
    mask = inv["make"] == make
    inv.loc[mask, "model_family"] = (
        inv.loc[mask, "model"].replace(mapping)
    )

# 4. Use the exact 90 vehicle groups in our dataset
vehicle_keys = ["make", "model_family", "model_year"]

vehicle_groups = (
    modeling_data[vehicle_keys]
    .drop_duplicates()
)

matched = inv.merge(
    vehicle_groups,
    on=vehicle_keys,
    how="inner",
    validate="many_to_one"
)

# 5. Rebuild broad event table
matched["action_type"] = (
    matched["action_number"].astype("string").str[:2]
)

broad_events = matched.loc[
    matched["action_type"].isin(
        ["PE", "EA", "RQ", "AQ", "DP"]
    ) &
    matched["open_date"].notna()
].copy()

broad_events = (
    broad_events
    .sort_values("open_date")
    .drop_duplicates(
        vehicle_keys + ["action_number"]
    )
    [
        vehicle_keys +
        ["action_number", "action_type",
         "open_date", "subject"]
    ]
    .reset_index(drop=True)
)

print("===== REBUILT INVESTIGATION EVENTS =====")
print("Event rows:", len(broad_events))
print("Unique actions:", broad_events["action_number"].nunique())
print(
    "Vehicle groups with actions:",
    broad_events[vehicle_keys].drop_duplicates().shape[0]
)

# 6. Save events for later auditing
output_path = ROOT / "data/processed/broad_events.csv"
broad_events.to_csv(output_path, index=False)

print("\nSaved:", output_path)

assert len(broad_events) == 104
assert broad_events["action_number"].nunique() == 33

print("Investigation event reconstruction passed!")


===== REBUILT INVESTIGATION EVENTS =====
Event rows: 104
Unique actions: 33
Vehicle groups with actions: 54

Saved: /Users/xinyisong/Desktop/DATA1030-Midterm-Xinfinity/data/processed/broad_events.csv
Investigation event reconstruction passed!


In [10]:

# A6.4.6 - Audit binary target labels

import pandas as pd

vehicle_keys = ["make", "model_family", "model_year"]
sample_keys = vehicle_keys + ["snapshot_date"]

# Build snapshot windows from the saved modeling dataset
audit_grid = modeling_data[
    sample_keys + ["future_investigation"]
].copy()

audit_grid["window_end"] = (
    audit_grid["snapshot_date"] +
    pd.DateOffset(months=12)
)

# Match investigation events by vehicle group
audit_matches = audit_grid.merge(
    broad_events[
        vehicle_keys + ["action_number", "open_date"]
    ],
    on=vehicle_keys,
    how="left"
)

# Check whether each action opens within the prediction window
audit_matches["event_in_window"] = (
    (audit_matches["open_date"] >=
     audit_matches["snapshot_date"]) &
    (audit_matches["open_date"] <
     audit_matches["window_end"])
)

# Recompute labels from the investigation event table
recomputed_labels = (
    audit_matches
    .groupby(sample_keys, as_index=False)
    .agg(
        recomputed_target=("event_in_window", "max")
    )
)

recomputed_labels["recomputed_target"] = (
    recomputed_labels["recomputed_target"].astype(int)
)

# Compare with originally saved targets
audit_result = audit_grid.merge(
    recomputed_labels,
    on=sample_keys,
    how="left",
    validate="one_to_one"
)

audit_result["target_match"] = (
    audit_result["future_investigation"] ==
    audit_result["recomputed_target"]
)

print("===== TARGET AUDIT =====")
print("Total samples:", len(audit_result))
print(
    "Original positives:",
    audit_result["future_investigation"].sum()
)
print(
    "Recomputed positives:",
    audit_result["recomputed_target"].sum()
)

mismatches = audit_result.loc[
    ~audit_result["target_match"]
]

print("Mismatched labels:", len(mismatches))

if not mismatches.empty:
    display(mismatches.head(20))

assert len(audit_result) == 2480
assert len(mismatches) == 0

print("\nTarget label audit passed!")


===== TARGET AUDIT =====
Total samples: 2480
Original positives: 357
Recomputed positives: 357
Mismatched labels: 0

Target label audit passed!


In [11]:

# A6.4.7 - Audit action-level event dependence

import pandas as pd

vehicle_keys = ["make", "model_family", "model_year"]

# Match events to existing sample windows
event_audit = audit_matches.loc[
    audit_matches["event_in_window"]
].copy()

# Attach split labels
event_audit = event_audit.merge(
    split_data[
        vehicle_keys + ["snapshot_date", "split"]
    ],
    on=vehicle_keys + ["snapshot_date"],
    how="left",
    validate="many_to_one"
)

print("===== ACTION DEPENDENCE BY SPLIT =====")

action_summary = (
    event_audit
    .groupby("split")
    .agg(
        positive_event_links=("action_number", "size"),
        unique_actions=("action_number", "nunique"),
        positive_snapshots=("snapshot_date", "size")
    )
)

display(action_summary)

print("\n===== MOST REPEATED ACTIONS =====")

action_repeats = (
    event_audit
    .groupby("action_number")
    .agg(
        snapshot_event_links=("action_number", "size"),
        affected_vehicle_groups=(
            "model_year", "nunique"
        )
    )
    .sort_values("snapshot_event_links", ascending=False)
)

display(action_repeats.head(10))

print("\n===== ACTION OVERLAP ACROSS SPLITS =====")

for split_name in ["train", "test"]:
    actions = set(
        event_audit.loc[
            event_audit["split"] == split_name,
            "action_number"
        ]
    )
    print(split_name, "unique actions:", len(actions))

train_actions = set(
    event_audit.loc[event_audit["split"] == "train",
                    "action_number"]
)

test_actions = set(
    event_audit.loc[event_audit["split"] == "test",
                    "action_number"]
)

overlap = train_actions & test_actions

print("Train/Test shared action numbers:", len(overlap))
print("Shared actions:", sorted(overlap))


===== ACTION DEPENDENCE BY SPLIT =====


,positive_event_links,unique_actions,positive_snapshots
split,,,
gap,32,7,32
test,29,4,29
train,334,28,334



===== MOST REPEATED ACTIONS =====


,snapshot_event_links,affected_vehicle_groups
action_number,,
DP22005,88,8
EA21002,52,4
PE21001,24,6
AQ22003,20,5
EA23001,16,4
EA19002,16,4
PE18013,16,4
PE18008,15,4
PE23023,12,3



===== ACTION OVERLAP ACROSS SPLITS =====
train unique actions: 28
test unique actions: 4
Train/Test shared action numbers: 0
Shared actions: []


In [12]:

# A6.4.8 - Detailed action dependence audit

vehicle_keys = ["make", "model_family", "model_year"]
sample_keys = vehicle_keys + ["snapshot_date"]

# 1. Correct statistics for each action
action_details = (
    event_audit
    .groupby(["split", "action_number"])
    .agg(
        snapshot_links=("snapshot_date", "size"),
        first_snapshot=("snapshot_date", "min"),
        last_snapshot=("snapshot_date", "max"),
        action_open_date=("open_date", "min")
    )
    .reset_index()
)

# Count unique complete vehicle groups
group_counts = (
    event_audit[
        ["split", "action_number"] + vehicle_keys
    ]
    .drop_duplicates()
    .groupby(["split", "action_number"])
    .size()
    .rename("unique_vehicle_groups")
    .reset_index()
)

action_details = action_details.merge(
    group_counts,
    on=["split", "action_number"],
    validate="one_to_one"
)

print("===== TEST ACTION DETAILS =====")

display(
    action_details.loc[
        action_details["split"] == "test"
    ].sort_values("snapshot_links", ascending=False)
)

# 2. Count distinct positive snapshots correctly
positive_snapshot_counts = (
    event_audit
    .groupby("split")
    .apply(
        lambda x: x[sample_keys].drop_duplicates().shape[0],
        include_groups=False
    )
    .rename("unique_positive_samples")
)

print("\n===== UNIQUE POSITIVE SAMPLES =====")
display(positive_snapshot_counts)

# 3. Check action overlap across all splits
action_sets = {
    name: set(
        event_audit.loc[
            event_audit["split"] == name,
            "action_number"
        ]
    )
    for name in ["train", "gap", "test"]
}

print("\n===== ACTION OVERLAP =====")

for a, b in [
    ("train", "gap"),
    ("gap", "test"),
    ("train", "test")
]:
    shared = action_sets[a] & action_sets[b]
    print(f"{a} / {b}: {len(shared)} shared")
    print(sorted(shared))

print("\nTotal unique actions across all splits:",
      len(set.union(*action_sets.values())))


===== TEST ACTION DETAILS =====


,split,action_number,snapshot_links,first_snapshot,last_snapshot,action_open_date,unique_vehicle_groups
9,test,PE25002,12,2024-04-01,2025-01-01,2025-03-21,3
7,test,EA26001,9,2025-04-01,2025-10-01,2026-01-30,3
8,test,PE24019,6,2024-01-01,2024-07-01,2024-07-29,2
10,test,RQ24008,2,2024-01-01,2024-04-01,2024-04-11,1



===== UNIQUE POSITIVE SAMPLES =====


split
gap       31
test      29
train    297
Name: unique_positive_samples, dtype: int64


===== ACTION OVERLAP =====
train / gap: 5 shared
['AQ23002', 'EA23002', 'EA23003', 'PE23013', 'PE23023']
gap / test: 2 shared
['PE24019', 'RQ24008']
train / test: 0 shared
[]

Total unique actions across all splits: 32


In [13]:

# A6.4.9 - Identify actions not linked to any positive snapshot

all_actions = set(
    broad_events["action_number"].dropna()
)

linked_actions = set(
    event_audit["action_number"].dropna()
)

unlinked_actions = sorted(
    all_actions - linked_actions
)

print("===== ACTION COVERAGE AUDIT =====")
print("All broad events actions:", len(all_actions))
print("Actions linked to snapshots:", len(linked_actions))
print("Unlinked actions:", len(unlinked_actions))
print("Unlinked action numbers:", unlinked_actions)

# Inspect events associated with these actions
unlinked_details = broad_events.loc[
    broad_events["action_number"].isin(unlinked_actions)
].sort_values(["action_number", "model_year"])

display(unlinked_details)

# Check test actions separately
print("\n===== TEST ACTION DETAILS =====")

test_action_details = action_details.loc[
    action_details["split"] == "test",
    [
        "action_number",
        "action_open_date",
        "snapshot_links",
        "unique_vehicle_groups"
    ]
].sort_values("snapshot_links", ascending=False)

display(test_action_details)


===== ACTION COVERAGE AUDIT =====
All broad events actions: 33
Actions linked to snapshots: 32
Unlinked actions: 1
Unlinked action numbers: ['EA26007']


,make,model_family,model_year,action_number,action_type,open_date,subject
103,HYUNDAI,ELANTRA,2016.0,EA26007,EA,2026-11-13,No Restart After Auto Start/Stop Engages



===== TEST ACTION DETAILS =====


,action_number,action_open_date,snapshot_links,unique_vehicle_groups
9,PE25002,2025-03-21,12,3
7,EA26001,2026-01-30,9,3
8,PE24019,2024-07-29,6,2
10,RQ24008,2024-04-11,2,1


In [14]:

# A6.5.1 - Inspect feature engineering code

keywords = [
    "filed_date",
    "complaints_last_12m",
    "complaints_last_3m",
    "complaint_trend_3m",
    "reporting_delay",
    "snapshot_date",
    "merge_asof"
]

for i, cell in enumerate(old_notebook["cells"]):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    # Feature engineering is located after target construction
    if i <= 45:
        continue

    matched = [
        keyword for keyword in keywords
        if keyword in source
    ]

    if matched:
        print(f"\n===== CELL {i} =====")
        print("Keywords:", matched)
        print(source[:3500])



===== CELL 46 =====
Keywords: ['filed_date', 'complaints_last_12m', 'snapshot_date']

# A5.1 - Count complaints in the previous 12 months

import pandas as pd

vehicle_keys = ["make", "model_family", "model_year"]

# Start from labeled snapshots
feature_grid = target_labels.copy()

# Ensure dates are datetime
complaints_features = complaints_final.copy()

complaints_features["filed_date"] = pd.to_datetime(
    complaints_features["filed_date"]
)

# Match complaints to their vehicle group
complaint_windows = feature_grid.merge(
    complaints_features[
        vehicle_keys + ["complaint_id", "filed_date"]
    ],
    on=vehicle_keys,
    how="left"
)

# Define the 12-month historical window
complaint_windows["history_start"] = (
    complaint_windows["snapshot_date"] -
    pd.DateOffset(months=12)
)

# Use only complaints submitted before snapshot date
complaint_windows["in_history"] = (
    (complaint_windows["filed_date"] >=
     complaint_windows["history_start"]) &
    (complaint_wi

In [15]:

# A6.5.2 - Find reporting delay construction

for i, cell in enumerate(old_notebook["cells"]):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if i >= 46:
        continue

    if "reporting_delay" in source:
        print(f"\n===== CELL {i} =====")
        print(source[:4000])



===== CELL 38 =====

# A3.21 - Inspect midterm reporting delay

print("Midterm columns:")
print(df.columns.tolist())

print("\nReporting delay sample:")
display(
    df[
        [
            "complaint_id",
            "filed_date",
            "incident_date",
            "reporting_delay",
            "invalid_incident_date"
        ]
    ].head(10)
)

print("\nReporting delay statistics:")
display(df["reporting_delay"].describe())

print("\nInvalid incident date values:")
display(
    df["invalid_incident_date"]
    .value_counts(dropna=False)
)


===== CELL 39 =====

# A3.22 - Find original midterm date cleaning logic

import nbformat

midterm_notebook = (
    ROOT / "src" / "preprocessing"
    / "00_shared_preprocessing.ipynb"
)

nb = nbformat.read(midterm_notebook, as_version=4)

keywords = [
    "reporting_delay",
    "invalid_incident_date",
    "incident_date"
]

for i, cell in enumerate(nb.cells):
    if cell.cell_type != "code":
        continue

    source = cell.source



In [16]:

# A6.5.3 - Reporting delay sanity check

from pathlib import Path
import pandas as pd

ROOT = Path(
    ROOT
)

complaints_check = pd.read_csv(
    ROOT / "data/processed/complaints_final.csv",
    parse_dates=["filed_date", "incident_date"]
)

delay = complaints_check["reporting_delay"]

print("===== REPORTING DELAY AUDIT =====")

print("Total complaints:", len(complaints_check))
print("Missing reporting delays:", delay.isna().sum())
print("Negative delays:", (delay < 0).sum())
print("Delays over 5 years:", (delay > 365 * 5).sum())

print("\n===== DELAY DISTRIBUTION =====")
display(
    delay.describe(
        percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
    )
)

print("\n===== NEGATIVE DELAY EXAMPLES =====")
display(
    complaints_check.loc[
        delay < 0,
        [
            "complaint_id",
            "model_year",
            "filed_date",
            "incident_date",
            "reporting_delay"
        ]
    ].head(10)
)

print("\n===== EXTREME DELAY EXAMPLES =====")
display(
    complaints_check.loc[
        delay > 365 * 5,
        [
            "complaint_id",
            "model_year",
            "filed_date",
            "incident_date",
            "reporting_delay"
        ]
    ].head(10)
)


===== REPORTING DELAY AUDIT =====
Total complaints: 39905
Missing reporting delays: 87
Negative delays: 0
Delays over 5 years: 209

===== DELAY DISTRIBUTION =====


count    39818.000000
mean       106.834748
std        274.288202
min          0.000000
1%           0.000000
5%           0.000000
50%         13.000000
95%        547.000000
99%       1455.660000
max       4448.000000
Name: reporting_delay, dtype: float64


===== NEGATIVE DELAY EXAMPLES =====


,complaint_id,model_year,filed_date,incident_date,reporting_delay



===== EXTREME DELAY EXAMPLES =====


,complaint_id,model_year,filed_date,incident_date,reporting_delay
8760,11385292,2015,2020-12-28,2015-12-15,1840.0
9302,11402831,2015,2021-03-13,2015-03-01,2204.0
9369,11404421,2015,2021-03-23,2016-03-01,1848.0
9429,11405720,2015,2021-03-31,2015-01-01,2281.0
9827,11418624,2016,2021-05-26,2016-05-26,1826.0
10613,11433591,2015,2021-09-20,2016-09-07,1839.0
10629,11433830,2015,2021-09-21,2015-04-23,2343.0
10985,11440847,2016,2021-11-17,2016-01-01,2147.0
11790,11455978,2015,2022-03-10,2016-01-15,2246.0
12405,11466242,2015,2022-05-26,2017-05-26,1826.0


In [17]:

# A6.5.4 - Final feature consistency audit

import numpy as np
import pandas as pd

df_check = modeling_data.copy()

print("===== FINAL FEATURE CONSISTENCY AUDIT =====")

# 1. Check duplicate sample keys
sample_keys = [
    "make", "model_family",
    "model_year", "snapshot_date"
]

duplicate_count = df_check.duplicated(sample_keys).sum()

print("Duplicate samples:", duplicate_count)

# 2. Verify complaint window ordering
invalid_window_order = (
    (df_check["complaints_last_3m"] >
     df_check["complaints_last_6m"]) |
    (df_check["complaints_last_6m"] >
     df_check["complaints_last_12m"])
).sum()

print("Invalid complaint window ordering:",
      invalid_window_order)

# 3. Severity and component counts should not exceed
#    the number of complaints in the same 12-month window
count_features = [
    "crash_count_12m",
    "fire_count_12m",
    "engine_complaints_12m",
    "brake_complaints_12m",
    "electrical_complaints_12m",
    "powertrain_complaints_12m",
    "airbag_complaints_12m",
    "valid_delay_count_12m"
]

for col in count_features:
    violations = (
        df_check[col] >
        df_check["complaints_last_12m"]
    ).sum()

    print(f"{col} violations:", violations)

# 4. Check rate boundaries
for col in ["crash_rate_12m", "fire_rate_12m"]:
    violations = (
        (df_check[col].dropna() < 0) |
        (df_check[col].dropna() > 1)
    ).sum()

    print(f"{col} invalid rates:", violations)

# 5. Check negative feature values
feature_cols = [
    col for col in df_check.columns
    if col not in sample_keys + ["future_investigation"]
]

numeric_features = df_check[feature_cols].select_dtypes(
    include="number"
)

negative_counts = (
    numeric_features < 0
).sum()

print("\n===== NEGATIVE FEATURES =====")
display(negative_counts[negative_counts > 0])

# 6. Check missing and infinite values
print("\n===== MISSING VALUES =====")
display(
    numeric_features.isna()
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(
    "\nInfinite values:",
    np.isinf(
        numeric_features.to_numpy(dtype=float)
    ).sum()
)

# 7. Confirm expected dataset structure
assert len(df_check) == 2480
assert df_check.shape[1] == 24
assert duplicate_count == 0
assert invalid_window_order == 0

print("\nBasic feature consistency checks passed!")


===== FINAL FEATURE CONSISTENCY AUDIT =====
Duplicate samples: 0
Invalid complaint window ordering: 0
crash_count_12m violations: 0
fire_count_12m violations: 0
engine_complaints_12m violations: 0
brake_complaints_12m violations: 0
electrical_complaints_12m violations: 0
powertrain_complaints_12m violations: 0
airbag_complaints_12m violations: 0
valid_delay_count_12m violations: 0
crash_rate_12m invalid rates: 0
fire_rate_12m invalid rates: 0

===== NEGATIVE FEATURES =====


Series([], dtype: int64)


===== MISSING VALUES =====


complaint_trend_3m            175
median_reporting_delay_12m     94
avg_reporting_delay_12m        94
crash_rate_12m                 93
fire_rate_12m                  93
brake_complaints_12m            0
component_diversity_12m         0
airbag_complaints_12m           0
powertrain_complaints_12m       0
electrical_complaints_12m       0
dtype: int64


Infinite values: 0

Basic feature consistency checks passed!


In [18]:

# A7.1 - Save final train and test datasets

from pathlib import Path
import pandas as pd

ROOT = Path(
    ROOT
)

output_dir = ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# Remove helper columns created for split validation
helper_cols = ["label_end", "split"]

train_final = (
    train_data
    .drop(columns=helper_cols, errors="ignore")
    .sort_values(
        ["snapshot_date", "make", "model_family", "model_year"]
    )
    .reset_index(drop=True)
)

test_final = (
    test_data
    .drop(columns=helper_cols, errors="ignore")
    .sort_values(
        ["snapshot_date", "make", "model_family", "model_year"]
    )
    .reset_index(drop=True)
)

# Save files
train_path = output_dir / "train.csv"
test_path = output_dir / "test.csv"

train_final.to_csv(train_path, index=False)
test_final.to_csv(test_path, index=False)

print("===== FINAL DATASETS =====")
print("Train:", train_final.shape)
print("Test:", test_final.shape)

print("\n===== POSITIVE LABELS =====")
print("Train:", train_final["future_investigation"].sum())
print("Test:", test_final["future_investigation"].sum())

# Validate no overlap and consistent columns
sample_keys = [
    "make", "model_family", "model_year", "snapshot_date"
]

assert train_final.shape == (1490, 24)
assert test_final.shape == (720, 24)

assert list(train_final.columns) == list(test_final.columns)

train_keys = pd.MultiIndex.from_frame(
    train_final[sample_keys]
)

test_keys = pd.MultiIndex.from_frame(
    test_final[sample_keys]
)

assert not train_keys.isin(test_keys).any()

assert train_final["snapshot_date"].max() < \
       test_final["snapshot_date"].min()

print("\nFinal datasets saved and validated!")
print("Train file:", train_path)
print("Test file:", test_path)


===== FINAL DATASETS =====
Train: (1490, 24)
Test: (720, 24)

===== POSITIVE LABELS =====
Train: 297
Test: 29

Final datasets saved and validated!
Train file: /Users/xinyisong/Desktop/DATA1030-Midterm-Xinfinity/data/processed/train.csv
Test file: /Users/xinyisong/Desktop/DATA1030-Midterm-Xinfinity/data/processed/test.csv


In [19]:

# A7.2.1 - Save feature configuration

import json
from pathlib import Path

ROOT = Path(
    ROOT
)

feature_cols = [
    col for col in train_final.columns
    if col not in [
        "make",
        "model_family",
        "model_year",
        "snapshot_date",
        "future_investigation"
    ]
]

config = {
    "target": "future_investigation",

    "sample_identifiers": [
        "make",
        "model_family",
        "model_year",
        "snapshot_date"
    ],

    "numeric_features": feature_cols,

    "primary_metric": "average_precision",

    "cv_strategy": "purged_expanding_window",

    "cv_folds": 3,

    "random_seeds": [42, 123, 2026],

    "notes": [
        "Fit preprocessing only on each training fold.",
        "Do not use the final test set for model selection.",
        "Do not randomly shuffle temporal samples.",
        "Preserve missing feature values before modeling.",
        "Use the saved CV fold assignments."
    ]
}

config_path = ROOT / "data/processed/model_config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)

print("===== FEATURE CONFIGURATION =====")
print("Number of numeric features:", len(feature_cols))
print("Target:", config["target"])
print("Primary metric:", config["primary_metric"])
print("CV strategy:", config["cv_strategy"])

print("\nSaved:", config_path)

assert len(feature_cols) == 19

print("\nFeature configuration saved successfully!")


===== FEATURE CONFIGURATION =====
Number of numeric features: 19
Target: future_investigation
Primary metric: average_precision
CV strategy: purged_expanding_window

Saved: /Users/xinyisong/Desktop/DATA1030-Midterm-Xinfinity/data/processed/model_config.json

Feature configuration saved successfully!


In [20]:

# A7.2.2 - Independent file loading and CV validation

import pandas as pd
import json
from pathlib import Path

ROOT = Path(
    ROOT
)
data_dir = ROOT / "data" / "processed"

# 1. Reload saved files
train_check = pd.read_csv(
    data_dir / "train.csv",
    parse_dates=["snapshot_date"]
)

test_check = pd.read_csv(
    data_dir / "test.csv",
    parse_dates=["snapshot_date"]
)

cv_check = pd.read_csv(
    data_dir / "cv_fold_assignments.csv",
    parse_dates=["snapshot_date"]
)

with open(data_dir / "model_config.json", "r") as f:
    config_check = json.load(f)

# 2. Validate feature configuration
feature_cols = config_check["numeric_features"]
target_col = config_check["target"]

assert len(feature_cols) == 19
assert all(col in train_check.columns for col in feature_cols)
assert all(col in test_check.columns for col in feature_cols)

# 3. Rebuild each CV fold using saved assignments
keys = [
    "make", "model_family",
    "model_year", "snapshot_date"
]

print("===== RELOADED CV FOLDS =====")

for fold_id in sorted(cv_check["fold"].unique()):

    fold_info = cv_check.loc[
        cv_check["fold"] == fold_id
    ]

    train_keys = fold_info.loc[
        fold_info["role"] == "train", keys
    ]

    vali_keys = fold_info.loc[
        fold_info["role"] == "vali", keys
    ]

    train_fold = train_check.merge(
        train_keys,
        on=keys,
        how="inner",
        validate="one_to_one"
    )

    vali_fold = train_check.merge(
        vali_keys,
        on=keys,
        how="inner",
        validate="one_to_one"
    )

    # Validate chronological separation, accounting for
    # the 12-month forward-looking target window
    last_train_label_end = (
        train_fold["snapshot_date"].max()
        + pd.DateOffset(months=12)
    )

    assert last_train_label_end <= \
           vali_fold["snapshot_date"].min()

    print(
        f"Fold {fold_id}: "
        f"Train={len(train_fold)}, "
        f"Vali={len(vali_fold)}, "
        f"Train positives={train_fold[target_col].sum()}, "
        f"Vali positives={vali_fold[target_col].sum()}"
    )

# 4. Final saved dataset checks
assert train_check.shape == (1490, 24)
assert test_check.shape == (720, 24)

assert train_check[target_col].sum() == 297
assert test_check[target_col].sum() == 29

print("\nIndependent loading and CV validation passed!")


===== RELOADED CV FOLDS =====
Fold 1: Train=410, Vali=240, Train positives=51, Vali positives=29
Fold 2: Train=620, Vali=280, Train positives=63, Vali positives=60
Fold 3: Train=870, Vali=320, Train positives=111, Vali positives=111

Independent loading and CV validation passed!
